# Lab 7: Give the assistant a tool
**Module 4.1: Tool calling and MCP.**  **Time:** 25 minutes.

**Goal:** see exactly how an LLM "uses" a tool. Key point: **the model never runs anything.**
It asks your code to run a function, and your code decides whether to do it.

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();
import json

## 1. A normal Python function (pretend it calls the order system's API)

In [ ]:
ORDERS = {
    "ORD-1001": {"status": "shipped", "eta": "5 Oct", "amount_inr": 2499},
    "ORD-1002": {"status": "delivered", "eta": None, "amount_inr": 7999},
    "ORD-1003": {"status": "processing", "eta": "9 Oct", "amount_inr": 18999},
}

def order_status(order_id: str) -> dict:
    """Look up an order in the order system."""
    return ORDERS.get(order_id.upper().strip(), {"error": f"order {order_id} not found"})

order_status("ORD-1001")

## 2. Describe it to the model
The description and parameter schema are all the model sees. Write them like API documentation.

In [ ]:
order_tool = lk.tool_spec(order_status, "Get the delivery status, ETA and amount of a customer order.",
                          {"order_id": {"type": "string", "description": "Order id, for example ORD-1001"}})
print(json.dumps(order_tool, indent=2))

## 3. The model asks for the tool

In [ ]:
messages = [{"role": "system", "content": "You are a support assistant. Use tools to look up facts."},
            {"role": "user", "content": "Hi, where is my order ORD-1001?"}]
reply = lk.chat(messages=messages, tools=[order_tool], tag="tool-1")
print("text content:", reply.content)
for call in reply.tool_calls or []:
    print("TOOL REQUESTED:", call.function.name, call.function.arguments)

## 4. Your code runs it and sends the result back

In [ ]:
call = reply.tool_calls[0]
result = order_status(**json.loads(call.function.arguments))
print("our code ran the tool, result:", result)

messages.append({"role": "assistant", "content": reply.content or "", "tool_calls": [
    {"id": call.id, "type": "function", "function": {"name": call.function.name, "arguments": call.function.arguments}}]})
messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

final = lk.chat(messages=messages, tag="tool-2")
print("FINAL ANSWER:", final)

## 5. No tool needed, no tool call

In [ ]:
r = lk.chat(messages=[{"role": "user", "content": "What are your support hours?"}], tools=[order_tool], tag="tool-3")
print("tool calls:", r.tool_calls, "\ntext:", r.content)

## 6. The same tool as an MCP server (read, do not run)
The **Model Context Protocol (MCP)** standardises steps 2 to 4 so any MCP-capable assistant or agent framework
can discover and call your tool. With the official Python SDK the order tool becomes:

```python
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("orders")

@mcp.tool()
def order_status(order_id: str) -> dict:
    """Get the delivery status, ETA and amount of a customer order."""
    return ORDERS.get(order_id, {"error": "not found"})

mcp.run()   # serves the tool over stdio or HTTP
```
The function and its docstring become the tool description automatically. The architecture question is
no longer "how do I call this API" but "who may call which MCP server, with what identity".

## Try this
1. Ask *"Compare orders ORD-1001 and ORD-1003"*. Does the model request two tool calls at once?
2. Make the description vague (`"order thing"`). Does the model still choose the tool?